In [1]:
import sqlite3

In [2]:
import pandas as pd

In [3]:
import os

In [4]:
METADATA_PATH = "/path/to/metadata/"
PATH_ChEMBL   = "/path/to/chembl_33/chembl_33.db"

In [6]:
wells   = pd.read_csv(METADATA_PATH+"metadata/well.csv.gz")

# Connect the database

In [7]:
con = sqlite3.connect(PATH_ChEMBL)

# Create a cursor

In [8]:
cur = con.cursor()

# Check the available tables

In [9]:
cur.execute("SELECT name FROM sqlite_master WHERE type='table';")
print(cur.fetchall())


[('action_type',), ('assay_type',), ('chembl_id_lookup',), ('confidence_score_lookup',), ('curation_lookup',), ('chembl_release',), ('source',), ('relationship_type',), ('target_type',), ('variant_sequences',), ('bioassay_ontology',), ('data_validity_lookup',), ('activity_smid',), ('activity_stds_lookup',), ('assay_classification',), ('atc_classification',), ('bio_component_sequences',), ('component_sequences',), ('protein_classification',), ('domains',), ('go_classification',), ('structural_alert_sets',), ('products',), ('frac_classification',), ('hrac_classification',), ('irac_classification',), ('research_stem',), ('organism_class',), ('patent_use_codes',), ('usan_stems',), ('version',), ('cell_dictionary',), ('docs',), ('target_dictionary',), ('tissue_dictionary',), ('molecule_dictionary',), ('activity_supp',), ('component_class',), ('component_domains',), ('component_go',), ('component_synonyms',), ('structural_alerts',), ('defined_daily_dose',), ('product_patents',), ('protein_cl

# Get all the assay data

In [10]:
command = """select * from assays"""

In [11]:

assay_df = pd.read_sql_query(command, con)

In [12]:
assay_df.head()

,assay_id,doc_id,description,assay_type,assay_test_type,assay_category,assay_organism,assay_tax_id,assay_strain,assay_tissue,...,confidence_score,curated_by,src_id,src_assay_id,chembl_id,cell_id,bao_format,tissue_id,variant_id,aidx
0,1,11087,The compound was tested for the in vitro inhib...,B,None,None,None,NaN,None,None,...,8,Autocuration,1,None,CHEMBL615117,NaN,BAO_0000019,NaN,NaN,CLD0
1,2,684,Compound was evaluated for its ability to mobi...,F,None,None,None,NaN,None,None,...,0,Autocuration,1,None,CHEMBL615118,NaN,BAO_0000219,NaN,NaN,CLD0
2,3,15453,None,B,None,None,None,NaN,None,None,...,0,Autocuration,1,None,CHEMBL615119,NaN,BAO_0000019,NaN,NaN,CLD0
3,4,17841,Binding affinity against A2 adenosine receptor...,B,None,None,Bos taurus,9913.0,None,Striatum,...,4,Autocuration,1,None,CHEMBL615120,NaN,BAO_0000249,2435.0,NaN,CLD0
4,5,17430,In vitro cell cytotoxicity against 143-B cell ...,F,None,None,Homo sapiens,9606.0,None,None,...,1,Intermediate,1,None,CHEMBL615121,163.0,BAO_0000219,NaN,NaN,CLD0


# Get all the compounds

In [13]:
command = """select * from compound_structures"""

In [14]:

compound_df = pd.read_sql_query(command, con)

In [15]:
compound_df.head()

,molregno,molfile,standard_inchi,standard_inchi_key,canonical_smiles
0,1,\n RDKit 2D\n\n 24 26 0 0 0 0...,InChI=1S/C17H12ClN3O3/c1-10-8-11(21-17(24)20-1...,OWRSAHYFSSNENM-UHFFFAOYSA-N,Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccccc1Cl
1,2,\n RDKit 2D\n\n 25 27 0 0 0 0...,InChI=1S/C18H12N4O3/c1-11-8-14(22-18(25)21-16(...,ZJYUMURGSZQFMH-UHFFFAOYSA-N,Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccc(C#N)cc1
2,3,\n RDKit 2D\n\n 25 27 0 0 0 0...,InChI=1S/C18H16ClN3O3/c1-10-7-14(22-18(25)21-1...,YOMWDCALSDWFSV-UHFFFAOYSA-N,Cc1cc(-n2ncc(=O)[nH]c2=O)cc(C)c1C(O)c1ccc(Cl)cc1
3,4,\n RDKit 2D\n\n 23 25 0 0 0 0...,InChI=1S/C17H13N3O3/c1-11-2-4-12(5-3-11)16(22)...,PSOPUAQFGCRDIP-UHFFFAOYSA-N,Cc1ccc(C(=O)c2ccc(-n3ncc(=O)[nH]c3=O)cc2)cc1
4,5,\n RDKit 2D\n\n 24 26 0 0 0 0...,InChI=1S/C17H12ClN3O3/c1-10-8-13(21-17(24)20-1...,KEZNSCMBVRNOHO-UHFFFAOYSA-N,Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccc(Cl)cc1


In [16]:
chembl_compounds = set(compound_df.standard_inchi_key.unique())
chembl_compounds_InChI = set(compound_df.standard_inchi_key.unique())

# Load all the JUMP-CP compounds

In [17]:
compond = pd.read_csv(METADATA_PATH+"metadata/compound.csv.gz")

In [18]:
compond.head()

,Metadata_JCP2022,Metadata_InChIKey,Metadata_InChI
0,JCP2022_000001,AAAHWCWPZPSPIW-UHFFFAOYSA-N,InChI=1S/C25H31N5O2/c1-4-23-26-14-16-30(23)24-...
1,JCP2022_000002,AAAJHRMBUHXWLD-UHFFFAOYSA-N,InChI=1S/C11H13ClN2O/c12-10-4-2-9(3-5-10)8-14-...
2,JCP2022_000003,AAALVYBICLMAMA-UHFFFAOYSA-N,InChI=1S/C20H15N3O2/c24-19-15-11-17(21-13-7-3-...
3,JCP2022_000004,AAANUZMCJQUYNX-UHFFFAOYSA-N,InChI=1S/C13H22N4O2S/c1-2-7-16-13(5-6-15-16)20...
4,JCP2022_000005,AAAQFGUYHFJNHI-UHFFFAOYSA-N,InChI=1S/C22H22ClN5O2/c1-4-24-20(29)12-18-22-2...


In [19]:
jump_compounds       = set(compond.Metadata_InChIKey.unique())
jump_compounds_InChI = set(compond.Metadata_InChI.unique())

# Find overlap in compounds


<div class="alert-success">
30171 compounds in common 
</div>

In [20]:
overlapping_compounds = chembl_compounds.intersection(jump_compounds)

# Get the unique_id of the compounds

In [21]:
df_overlap_compounds = compound_df[compound_df.standard_inchi_key.isin(overlapping_compounds)]

In [22]:
molregno_overlapping = df_overlap_compounds.molregno.values

In [23]:
df_overlap_compounds

,molregno,molfile,standard_inchi,standard_inchi_key,canonical_smiles
75,97,\n RDKit 2D\n\n 28 31 0 0 0 0...,InChI=1S/C19H21N5O4/c1-26-15-10-12-13(11-16(15...,IENZQIKPVFGBNW-UHFFFAOYSA-N,COc1cc2nc(N3CCN(C(=O)c4ccco4)CC3)nc(N)c2cc1OC
122,146,\n RDKit 2D\n\n 26 29 0 0 0 0...,InChI=1S/C18H20FN3O4/c1-10-9-26-17-14-11(16(23...,GSDSWSVVBLHKDQ-UHFFFAOYSA-N,CC1COc2c(N3CCN(C)CC3)c(F)cc3c(=O)c(C(=O)O)cn1c23
123,147,\n RDKit 2D\n\n 17 18 0 0 0 0...,InChI=1S/C12H12N2O3/c1-3-14-6-9(12(16)17)10(15...,MHWLWQUZZRMNGJ-UHFFFAOYSA-N,CCn1cc(C(=O)O)c(=O)c2ccc(C)nc21
130,154,\n RDKit 2D\n\n 47 51 0 0 0 0...,InChI=1S/C36H38F4N4O2S/c1-3-42(4-2)20-21-43(22...,WDPFJWLDPVQCAJ-UHFFFAOYSA-N,CCN(CC)CCN(Cc1ccc(-c2ccc(C(F)(F)F)cc2)cc1)C(=O...
148,173,\n RDKit 2D\n\n 25 27 0 0 0 0...,InChI=1S/C19H16ClNO4/c1-11-15(10-18(22)23)16-9...,CGIGDMFJXJATDK-UHFFFAOYSA-N,COc1ccc2c(c1)c(CC(=O)O)c(C)n2C(=O)c1ccc(Cl)cc1
...,...,...,...,...,...
2369195,2781953,\n RDKit 2D\n\n 16 17 0 0 0 ...,InChI=1S/C11H13N3OS/c1-7-4-11(13-6-12-7)16-5-1...,ROKHJFVYDKKVDA-UHFFFAOYSA-N,Cc1cc(SCc2c(C)noc2C)ncn1
2369709,2782471,\n RDKit 2D\n\n 15 17 0 0 0 ...,InChI=1S/C11H9N3O/c1-7-11(14-15-13-7)10-6-8-4-...,QDSCGWAUHTUMSL-UHFFFAOYSA-N,Cc1nonc1-c1cc2ccccc2[nH]1
2369982,2782744,\n RDKit 2D\n\n 18 20 0 0 0 ...,InChI=1S/C11H12N4O2S/c1-5-8(6(2)17-15-5)9(16)1...,OXBPDHNXDPMVNB-UHFFFAOYSA-N,Cc1noc(C)c1C(=O)Nc1nnc(C2CC2)s1
2369989,2782751,\n RDKit 2D\n\n 30 34 0 0 0 ...,InChI=1S/C25H21N3O2/c29-25(28-13-15-30-16-14-2...,XMOAXXRVOGXRCI-UHFFFAOYSA-N,O=C(c1ccc2nc(-c3ccccc3)c(-c3ccccc3)nc2c1)N1CCOCC1


# Check activity data

In [24]:
command = """select * from activities"""

In [25]:
activity_df = pd.read_sql_query(command, con)

In [26]:
activity_df.head()

,activity_id,assay_id,doc_id,record_id,molregno,standard_relation,standard_value,standard_units,standard_flag,standard_type,...,upper_value,standard_upper_value,src_id,type,relation,value,units,text_value,standard_text_value,action_type
0,31863,54505,6424,206172,180094,>,100000.0,nM,1,IC50,...,NaN,None,1,IC50,>,100.0,uM,None,None,None
1,31864,83907,6432,208970,182268,=,2500.0,nM,1,IC50,...,NaN,None,1,IC50,=,2.5,uM,None,None,None
2,31865,88152,6432,208970,182268,>,50000.0,nM,1,IC50,...,NaN,None,1,IC50,>,50.0,uM,None,None,None
3,31866,83907,6432,208987,182855,=,9000.0,nM,1,IC50,...,NaN,None,1,IC50,=,9.0,uM,None,None,None
4,31867,88153,6432,208987,182855,None,NaN,nM,0,IC50,...,NaN,None,1,IC50,None,NaN,uM,None,None,None


In [27]:
activity_df.shape

(20334684, 28)


<div class="alert-success">
20 million different activity read outs

The different types seen below
</div>

In [28]:
activity_df.standard_type.value_counts().head(50)

Potency                                 4473542
IC50                                    2663617
GI50                                    2618475
Inhibition                              1458213
Percent Effect                          1328366
Activity                                1205159
Ki                                       761069
MIC                                      706595
EC50                                     500081
INHIBITION                               339133
Kd                                       179399
AC50                                     156967
Z score                                  141783
Tissue Severity Score                    128999
Ratio IC50                               126714
GI                                       125357
Ratio                                    115190
ED50                                     102764
IZ                                        93990
CC50                                      93661
T1/2                                    

# Find the activity readouts of the overlapping compounds

In [29]:
activity_df.head()

,activity_id,assay_id,doc_id,record_id,molregno,standard_relation,standard_value,standard_units,standard_flag,standard_type,...,upper_value,standard_upper_value,src_id,type,relation,value,units,text_value,standard_text_value,action_type
0,31863,54505,6424,206172,180094,>,100000.0,nM,1,IC50,...,NaN,None,1,IC50,>,100.0,uM,None,None,None
1,31864,83907,6432,208970,182268,=,2500.0,nM,1,IC50,...,NaN,None,1,IC50,=,2.5,uM,None,None,None
2,31865,88152,6432,208970,182268,>,50000.0,nM,1,IC50,...,NaN,None,1,IC50,>,50.0,uM,None,None,None
3,31866,83907,6432,208987,182855,=,9000.0,nM,1,IC50,...,NaN,None,1,IC50,=,9.0,uM,None,None,None
4,31867,88153,6432,208987,182855,None,NaN,nM,0,IC50,...,NaN,None,1,IC50,None,NaN,uM,None,None,None


In [30]:
activity_of_overlapping_compounds = activity_df[activity_df.molregno.isin(molregno_overlapping)]

## Subset of compound in image dataset and corresponding activity readouts

In [31]:
activity_of_overlapping_compounds.head()

,activity_id,assay_id,doc_id,record_id,molregno,standard_relation,standard_value,standard_units,standard_flag,standard_type,...,upper_value,standard_upper_value,src_id,type,relation,value,units,text_value,standard_text_value,action_type
90,31953,171186,7018,110557,99368,=,20.000,mg kg-1 day-1,0,Dose,...,NaN,None,1,Dose,=,20.000,mg kg-1 day-1,None,None,None
91,31954,81863,7018,110557,99368,=,100.000,mg kg-1 day-1,0,Dose,...,NaN,None,1,Dose,=,100.000,mg kg-1 day-1,None,None,None
228,32091,138231,3408,190478,80299,=,0.317,nM,1,Kb,...,NaN,None,1,Kb,=,0.317,nM,None,None,None
229,32092,139983,3408,190478,80299,=,4.130,nM,1,Kb,...,NaN,None,1,Kb,=,4.130,nM,None,None,None
230,32093,229805,3408,190478,80299,=,13.000,None,0,Ratio,...,NaN,None,1,Ratio,=,13.000,None,None,None,None


In [32]:
activity_of_overlapping_compounds

,activity_id,assay_id,doc_id,record_id,molregno,standard_relation,standard_value,standard_units,standard_flag,standard_type,...,upper_value,standard_upper_value,src_id,type,relation,value,units,text_value,standard_text_value,action_type
90,31953,171186,7018,110557,99368,=,20.000,mg kg-1 day-1,0,Dose,...,NaN,None,1,Dose,=,20.000,mg kg-1 day-1,None,None,None
91,31954,81863,7018,110557,99368,=,100.000,mg kg-1 day-1,0,Dose,...,NaN,None,1,Dose,=,100.000,mg kg-1 day-1,None,None,None
228,32091,138231,3408,190478,80299,=,0.317,nM,1,Kb,...,NaN,None,1,Kb,=,0.317,nM,None,None,None
229,32092,139983,3408,190478,80299,=,4.130,nM,1,Kb,...,NaN,None,1,Kb,=,4.130,nM,None,None,None
230,32093,229805,3408,190478,80299,=,13.000,None,0,Ratio,...,NaN,None,1,Ratio,=,13.000,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20334567,24971595,2263402,124914,3901988,1280,None,NaN,None,0,EC50,...,NaN,None,1,EC50,None,NaN,None,None,None,None
20334583,24971611,2263410,124914,3901991,25497,=,7.000,nM,1,Ki,...,NaN,None,1,Ki,=,0.007,uM,None,None,AGONIST
20334591,24971619,2263412,124914,3901993,56133,=,1.000,nM,1,Ki,...,NaN,None,1,Ki,=,0.001,uM,None,None,AGONIST
20334596,24971624,2263414,124914,3901991,25497,=,20.000,nM,1,EC50,...,NaN,None,1,EC50,=,0.020,uM,None,None,PARTIAL AGONIST



<div class="alert-success">
1,354,041 million different activity read outs for the overlapping compounds

The different types seen below
</div>

In [33]:
setting = {"standard_type" : "Potency"}

# Selection methods:

In [34]:
activity_of_overlapping_compounds.standard_type.value_counts()[:10]

Potency                  285268
IC50                     170210
Ki                        77462
Kd                        73429
Activity                  69219
Tissue Severity Score     56964
Inhibition                55125
MIC                       33927
GI50                      30453
Residual Activity         29016
Name: standard_type, dtype: int64


<div class="alert-success">
285,268 different Potency activity read outs for the overlapping compounds

</div>

## Using Potency all

In [51]:
with_known_relation = True # if only using those with defined relation
potency_subset = activity_of_overlapping_compounds[activity_of_overlapping_compounds.standard_type == "Potency"]
potency_subset = potency_subset[potency_subset.activity_comment.isin(['inactive', 'active', 'Active', 'Not Active'])]
compound_counts = potency_subset.assay_id.value_counts()

selected_subset = potency_subset[potency_subset.assay_id.isin(compound_counts[(compound_counts > 100)].index)]

In [53]:
selected_subset.loc[:,"activity_based_on_comment"] = 0

/home/jhaslum/miniconda3/lib/python3.8/site-packages/pandas/core/indexing.py:1667: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.obj[key] = value


In [54]:
selected_subset.loc[selected_subset.activity_comment.isin(["Active", "active"]), "activity_based_on_comment"] = 1

/home/jhaslum/miniconda3/lib/python3.8/site-packages/pandas/core/indexing.py:1817: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_single_column(loc, value, pi)


In [55]:
selected_subset.loc[selected_subset.activity_comment.isin(["Not Active", "inactive"]), "activity_based_on_comment"] = -1

In [56]:
selected_subset.activity_based_on_comment.value_counts()

-1    60256
 1    37970
Name: activity_based_on_comment, dtype: int64

In [57]:
import numpy as np

In [58]:
label_matrix = selected_subset.pivot_table(values="activity_based_on_comment", index="molregno",columns="assay_id", aggfunc=np.median)

In [59]:
known = label_matrix.isna() == 0
label_matrix = label_matrix.fillna(0)

## Reduce size to keep assay meeting criteria

### Get subset of assays that meet criteria

In [ ]:
assay_ids = ((label_matrix == 1).sum() > 50)
subset_of_assays = assay_ids[assay_ids].index
subset_with_sufficent_negatives = ((label_matrix[subset_of_assays] == -1.0).sum() > 50)
subset_of_assays_to_keep = subset_with_sufficent_negatives[subset_with_sufficent_negatives].index

### Select that subset

In [ ]:
label_matrix_subset = label_matrix[subset_of_assays_to_keep]
known = (label_matrix_subset != 0.0).sum(axis=1)

### Select subset with overlap between activity data and JUMP-CP source-11

In [62]:
selected_subset = compound_df[compound_df.molregno.isin(label_matrix_subset.index)]

subset_of_compounds_JUMP_id = compond[compond.Metadata_InChIKey.isin(selected_subset.standard_inchi_key)].Metadata_JCP2022.values

In [81]:
well_subset_ = wells[wells.Metadata_JCP2022.isin(subset_of_compounds_JUMP_id)]
subset_of_compouns_in_source_JCP = well_subset_[well_subset_.Metadata_Source == "source_11"].Metadata_JCP2022.values

source_11_molregno = compound_df[compound_df.standard_inchi_key.isin(compond[compond.Metadata_JCP2022.isin(subset_of_compouns_in_source_JCP)].Metadata_InChIKey.values)].molregno.values

In [83]:
well_subset_

,Metadata_Source,Metadata_Plate,Metadata_Well,Metadata_JCP2022
1,source_1,UL000081,A03,JCP2022_085227
5,source_1,UL000081,A07,JCP2022_019918
15,source_1,UL000081,A17,JCP2022_035042
18,source_1,UL000081,A20,JCP2022_086636
32,source_1,UL000081,A34,JCP2022_089988
...,...,...,...,...
1096033,source_9,GR00004421,Z08,JCP2022_050165
1096047,source_9,GR00004421,Z22,JCP2022_041333
1096049,source_9,GR00004421,Z24,JCP2022_037716
1096050,source_9,GR00004421,Z25,JCP2022_037716


In [87]:
positive_ids = ((label_matrix_subset.loc[source_11_molregno] == 1.0).sum() > 50)
negative_ids = ((label_matrix_subset[positive_ids[positive_ids].index].loc[source_11_molregno] == -1.0).sum() > 50)

label_matrix_filtered = label_matrix_subset[negative_ids[negative_ids].index].loc[source_11_molregno]


In [90]:
label_matrix_filtered.reset_index()
label_matrix_renamed = label_matrix_filtered.merge(df_overlap_compounds[["molregno", "standard_inchi_key"]],
                            how="left", left_on="molregno", right_on="molregno",)

In [91]:
wells_11      = wells[wells.Metadata_Source == "source_11"]
wells_11_meta = wells_11.merge(compond, how="left", on="Metadata_JCP2022")
wells_11_meta_overlapping = wells_11_meta[wells_11_meta.Metadata_InChIKey.isin(label_matrix_renamed.standard_inchi_key.unique())]

combined = wells_11_meta_overlapping.merge(label_matrix_renamed, how = "left", right_on = "standard_inchi_key", left_on = "Metadata_InChIKey")

In [92]:
combined

,Metadata_Source,Metadata_Plate,Metadata_Well,Metadata_JCP2022,Metadata_InChIKey,Metadata_InChI,molregno,688128,688238,688360,...,845045,845102,845164,845169,845173,845177,845196,954338,1495346,standard_inchi_key
0,source_11,EC000001,A01,JCP2022_085227,SRVFFFJZQVENJC-UHFFFAOYSA-N,InChI=1S/C17H30N2O5/c1-6-23-17(22)14-13(24-14)...,1532209,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,SRVFFFJZQVENJC-UHFFFAOYSA-N
1,source_11,EC000001,A18,JCP2022_055168,MMVYAKGBFSWVAV-UHFFFAOYSA-N,InChI=1S/C22H26N2O/c1-16-10-12-18(13-11-16)21(...,848238,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,MMVYAKGBFSWVAV-UHFFFAOYSA-N
2,source_11,EC000001,A24,JCP2022_035095,IHLVSLOZUHKNMQ-UHFFFAOYSA-N,InChI=1S/C26H27N5O2/c1-2-9-27-22(4-1)26-25(24-...,429222,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,IHLVSLOZUHKNMQ-UHFFFAOYSA-N
3,source_11,EC000001,B01,JCP2022_037716,IVUGFMLRJOCGAS-UHFFFAOYSA-N,InChI=1S/C28H21N7OS/c1-17-15-24(37-16-17)25-20...,1412359,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,IVUGFMLRJOCGAS-UHFFFAOYSA-N
4,source_11,EC000001,B14,JCP2022_018463,DVITUKJKEPTQAE-UHFFFAOYSA-N,InChI=1S/C24H25N3O2/c1-18-7-3-6-10-23(18)29-16...,937623,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,DVITUKJKEPTQAE-UHFFFAOYSA-N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25905,source_11,LM71-102_2,P13,JCP2022_010654,CEUORZQYGODEFX-UHFFFAOYSA-N,InChI=1S/C23H27Cl2N3O2/c24-19-4-3-5-21(23(19)2...,155006,0.0,0.0,0.0,...,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,CEUORZQYGODEFX-UHFFFAOYSA-N
25906,source_11,LM71-102_2,P16,JCP2022_030831,HKQYGTCOTHHOMP-UHFFFAOYSA-N,InChI=1S/C16H12O4/c1-19-12-5-2-10(3-6-12)14-9-...,391222,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,HKQYGTCOTHHOMP-UHFFFAOYSA-N
25907,source_11,LM71-102_2,P19,JCP2022_096067,VSWDORGPIHIGNW-UHFFFAOYSA-N,"InChI=1S/C5H9NS2/c7-5(8)6-3-1-2-4-6/h1-4H2,(H,...",421017,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,VSWDORGPIHIGNW-UHFFFAOYSA-N
25908,source_11,LM71-102_2,P20,JCP2022_032357,HSUGRBWQSSZJOP-UHFFFAOYSA-N,InChI=1S/C22H26N2O4S/c1-15(25)28-20-21(16-9-11...,207725,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,HSUGRBWQSSZJOP-UHFFFAOYSA-N


In [93]:
wells_11

,Metadata_Source,Metadata_Plate,Metadata_Well,Metadata_JCP2022
165848,source_11,DMSO71-102_1,A01,JCP2022_033924
165849,source_11,DMSO71-102_1,A02,JCP2022_033924
165850,source_11,DMSO71-102_1,A03,JCP2022_033924
165851,source_11,DMSO71-102_1,A04,JCP2022_033924
165852,source_11,DMSO71-102_1,A05,JCP2022_033924
...,...,...,...,...
234190,source_11,LM71-102_2,P20,JCP2022_032357
234191,source_11,LM71-102_2,P21,JCP2022_047545
234192,source_11,LM71-102_2,P22,JCP2022_043099
234193,source_11,LM71-102_2,P23,JCP2022_050516


In [96]:
wells_11.loc[:,"sample"] = 0
df_sites = pd.DataFrame([1,2,3,4,5,6,7,8,9], columns=["Metadata_Site"])
df_sites.loc[:,"sample"] = 0
well_11_with_sites = wells_11[["Metadata_Plate", "Metadata_Well", "sample"]].merge(df_sites)

well_11_with_sites["Metadata_Path"] = well_11_with_sites["Metadata_Plate"] + "/" + well_11_with_sites["Metadata_Well"] + "_" + well_11_with_sites["Metadata_Site"].astype(str)  + ".png"

files_df = well_11_with_sites

merged_with_wells = combined.merge(files_df, how="left", on = ["Metadata_Plate", "Metadata_Well"])

/home/jhaslum/miniconda3/lib/python3.8/site-packages/pandas/core/indexing.py:1667: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.obj[key] = value


In [97]:
merged_with_wells

,Metadata_Source,Metadata_Plate,Metadata_Well,Metadata_JCP2022,Metadata_InChIKey,Metadata_InChI,molregno,688128,688238,688360,...,845169,845173,845177,845196,954338,1495346,standard_inchi_key,sample,Metadata_Site,Metadata_Path
0,source_11,EC000001,A01,JCP2022_085227,SRVFFFJZQVENJC-UHFFFAOYSA-N,InChI=1S/C17H30N2O5/c1-6-23-17(22)14-13(24-14)...,1532209,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,SRVFFFJZQVENJC-UHFFFAOYSA-N,0,1,EC000001/A01_1.png
1,source_11,EC000001,A01,JCP2022_085227,SRVFFFJZQVENJC-UHFFFAOYSA-N,InChI=1S/C17H30N2O5/c1-6-23-17(22)14-13(24-14)...,1532209,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,SRVFFFJZQVENJC-UHFFFAOYSA-N,0,2,EC000001/A01_2.png
2,source_11,EC000001,A01,JCP2022_085227,SRVFFFJZQVENJC-UHFFFAOYSA-N,InChI=1S/C17H30N2O5/c1-6-23-17(22)14-13(24-14)...,1532209,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,SRVFFFJZQVENJC-UHFFFAOYSA-N,0,3,EC000001/A01_3.png
3,source_11,EC000001,A01,JCP2022_085227,SRVFFFJZQVENJC-UHFFFAOYSA-N,InChI=1S/C17H30N2O5/c1-6-23-17(22)14-13(24-14)...,1532209,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,SRVFFFJZQVENJC-UHFFFAOYSA-N,0,4,EC000001/A01_4.png
4,source_11,EC000001,A01,JCP2022_085227,SRVFFFJZQVENJC-UHFFFAOYSA-N,InChI=1S/C17H30N2O5/c1-6-23-17(22)14-13(24-14)...,1532209,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,SRVFFFJZQVENJC-UHFFFAOYSA-N,0,5,EC000001/A01_5.png
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
233185,source_11,LM71-102_2,P22,JCP2022_043099,JZFPYUNJRRFVQU-UHFFFAOYSA-N,"InChI=1S/C13H9F3N2O2/c14-13(15,16)8-3-1-4-9(7-...",97010,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,JZFPYUNJRRFVQU-UHFFFAOYSA-N,0,5,LM71-102_2/P22_5.png
233186,source_11,LM71-102_2,P22,JCP2022_043099,JZFPYUNJRRFVQU-UHFFFAOYSA-N,"InChI=1S/C13H9F3N2O2/c14-13(15,16)8-3-1-4-9(7-...",97010,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,JZFPYUNJRRFVQU-UHFFFAOYSA-N,0,6,LM71-102_2/P22_6.png
233187,source_11,LM71-102_2,P22,JCP2022_043099,JZFPYUNJRRFVQU-UHFFFAOYSA-N,"InChI=1S/C13H9F3N2O2/c14-13(15,16)8-3-1-4-9(7-...",97010,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,JZFPYUNJRRFVQU-UHFFFAOYSA-N,0,7,LM71-102_2/P22_7.png
233188,source_11,LM71-102_2,P22,JCP2022_043099,JZFPYUNJRRFVQU-UHFFFAOYSA-N,"InChI=1S/C13H9F3N2O2/c14-13(15,16)8-3-1-4-9(7-...",97010,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,JZFPYUNJRRFVQU-UHFFFAOYSA-N,0,8,LM71-102_2/P22_8.png


# Metadata and labels 

In [555]:
merged_with_wells.to_csv("metadata_and_label_matrix_comment.csv")